# Classical control v2 (no neural network): identifiability of Cf from h-only data
**Version 2 — what changed after the first Kaggle run (cl9-0709):**
- statistics: several noise realisations / observation layouts per configuration, *principled* equivalence test (χ² ≤ χ²(true params) + 2·√(2/N)) and an explicit **identified / not-identified / optimiser-failed** verdict per realisation;
- the failed row of run 1 (zonal, U0=1, h+u+v, χ²=2.04) is addressed by more starts, tight tolerances and the verdict above (a start that does not reach noise level is reported as *optimiser failure*, never as an information result);
- free field: **λ continuation sweep** (large→small, warm-started), discrepancy-principle choice of λ (uses no truth), convergence flags;
- Jacobian: the meaningless `rank(1e-6)` / `cond` columns are removed (cell (0,0) of Cf never enters the solver → one exact null direction); only the noise-normalised **information modes** are reported;
- **new: identifiability–complexity curve (H1d) prediction** — Cramér–Rao bounds for a K-zone model, K = 1,2,3,4,6, from the forward Jacobian (no fitting, cheap).

Everything runs on CPU. Choose `MODE` in Cell 1. First run `MODE="quick"` (~6 min) only to check the pipeline, then `"standard"`.
Authoring status: the JAX-specific constructs are the same ones that executed correctly on Kaggle in run 1; new logic was exercised with a NumPy mock of JAX. Cell 3 (gate) must pass before anything is interpreted.

In [1]:
import os, json, time
import numpy as np
import jax, jax.numpy as jnp
from jax import lax
from scipy.optimize import minimize
jax.config.update("jax_enable_x64", True)

MODE = "standard"      # "quick" (~6 min, pipeline check) | "standard" (~25-40 min) | "full" (~1-1.5 h)   [time estimates, not measurements]
TEST_MODE = False      # authoring-sandbox smoke only; leave False on Kaggle

_MODES = {
    #            starts  zonal_it  free_it  lambdas (descending is applied internally)       noise_zonal  noise_free
    "quick":    dict(N_STARTS=3, ZONAL_MAXITER=60,  FREE_MAXITER=120, LAMBDAS=[1e-2, 1e-3],               N_NOISE=1, N_NOISE_FREE=1),
    "standard": dict(N_STARTS=6, ZONAL_MAXITER=120, FREE_MAXITER=250, LAMBDAS=[1e-1, 1e-2, 1e-3, 1e-4],    N_NOISE=3, N_NOISE_FREE=1),
    "full":     dict(N_STARTS=8, ZONAL_MAXITER=150, FREE_MAXITER=300, LAMBDAS=[1e0, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5], N_NOISE=5, N_NOISE_FREE=2),
}
CFG = dict(
    N_STEPS=1500, T0=150, T_STRIDE=25,        # 25 h = ~2 M2 cycles; observe after 2.5 h spin-up, every 25 min
    U0_LIST=[0.15, 1.0],                      # weak (Phase-0 twin) and strong forcing
    P_OBS=39,                                 # divisible by 3 -> exact equal-count comparison: h (39 pts) vs h+u+v (13 pts x 3 channels)
    NOISE_FRAC=0.03, SEED=7, CF_SCALE=2.5e-3,
    K_LIST=[1, 2, 3, 4, 6],
    TOL_BND_KM=6.25,                       # identified := boundary error <= 1 cell (6.25 km) ...
    TOL_CF_REL=0.25,                          # ... and both Cf values within 25 %
)
CFG.update(_MODES[MODE])
if TEST_MODE:
    CFG.update(N_STEPS=60, T0=20, T_STRIDE=10, N_STARTS=2, ZONAL_MAXITER=2, FREE_MAXITER=1, LAMBDAS=[1e-2],
               N_NOISE=1, N_NOISE_FREE=1, U0_LIST=[1.0], K_LIST=[1, 2, 3])
REGIMES = {"rich": np.arange(CFG["T0"], CFG["N_STEPS"], CFG["T_STRIDE"]),
           "sparse4": np.linspace(CFG["N_STEPS"] // 4, CFG["N_STEPS"] - 1, 4).astype(int)}   # like the PINN twin (4 snapshots)
if TEST_MODE: REGIMES = {"rich": REGIMES["rich"]}
print("MODE:", MODE, "| devices:", jax.devices())
print({k: v for k, v in CFG.items()})
print({k: len(v) for k, v in REGIMES.items()}, "snapshots per regime")

MODE: standard | devices: [CpuDevice(id=0)]
{'N_STEPS': 1500, 'T0': 150, 'T_STRIDE': 25, 'U0_LIST': [0.15, 1.0], 'P_OBS': 39, 'NOISE_FRAC': 0.03, 'SEED': 7, 'CF_SCALE': 0.0025, 'K_LIST': [1, 2, 3, 4, 6], 'TOL_BND_KM': 6.25, 'TOL_CF_REL': 0.25, 'N_STARTS': 6, 'ZONAL_MAXITER': 120, 'FREE_MAXITER': 250, 'LAMBDAS': [0.1, 0.01, 0.001, 0.0001], 'N_NOISE': 3, 'N_NOISE_FREE': 1}
{'rich': 54, 'sparse4': 4} snapshots per regime


In [2]:
# ---- core physics + problem definition (xp-generic: numpy for truth/tests, jax.numpy for inversion) ----
LX, LY = 150e3, 50e3
NX, NY = 24, 12
DX, DY = LX / NX, LY / NY
G, H0, F0, DT = 9.81, 80.0, 6.5e-5, 60.0
OMEGA = 2 * np.pi / (12.42 * 3600.0)
CF_LOW_TRUE, CF_HIGH_TRUE = 1.5e-3, 4.0e-3
B0_TRUE, S_TRUE = 0.5, 0.1          # boundary x/LX = B0 + S*(y/LY - 0.5)  (identical to the phase-0 twin)
XC = (np.arange(NX) + 0.5) * DX / LX     # normalized cell-centre coordinates
YC = (np.arange(NY) + 0.5) * DY / LY

def swe_step(xp, state, t, Cf, U0):
    eta, u, v = state
    vp = xp.pad(v, ((1, 1), (0, 0)), mode="edge"); vc = 0.5 * (vp[:, :-1] + vp[:, 1:]); v_at_u = 0.5 * (vc[:-1, :] + vc[1:, :])
    uc = 0.5 * (u[:-1, :] + u[1:, :]); up = xp.pad(uc, ((0, 0), (1, 1)), mode="edge"); u_at_v = 0.5 * (up[:, :-1] + up[:, 1:])
    Cf_u = Cf[np.clip(np.arange(NX + 1), 0, NX - 1), :]
    Cf_v = Cf[:, np.clip(np.arange(NY + 1), 0, NY - 1)]
    su = xp.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = xp.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    de_dx = (eta[1:, :] - eta[:-1, :]) / DX; de_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fu = Cf_u / H0 * u * su; fv = Cf_v / H0 * v * sv
    un = u[1:-1, :] + DT * (-G * de_dx + F0 * v_at_u[1:-1, :] - fu[1:-1, :])
    u_new = xp.concatenate([xp.full((1, NY), U0 * xp.cos(OMEGA * t)), un, xp.zeros((1, NY))], axis=0)
    vn = v[:, 1:-1] + DT * (-G * de_dy - F0 * u_at_v[:, 1:-1] - fv[:, 1:-1])
    v_new = xp.concatenate([xp.zeros((NX, 1)), vn, xp.zeros((NX, 1))], axis=1)
    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    return (eta - DT * H0 * div, u_new, v_new)

def run_obs_np(Cf, U0, n_steps, ox, oy):
    """numpy reference run -> (h,u,v) at observed cells for every step: 3 arrays (n_steps, P)."""
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H, U, V = [], [], []
    for n in range(n_steps):
        s = swe_step(np, s, n * DT, Cf, U0)
        e, u, v = s
        H.append(e[ox, oy]); U.append((0.5 * (u[:-1] + u[1:]))[ox, oy]); V.append((0.5 * (v[:, :-1] + v[:, 1:]))[ox, oy])
    return np.array(H), np.array(U), np.array(V)

def cf_zonal(xp, p, w_cells=0.5):
    """p = [ln Cf_low, ln Cf_high, b0, s]; smooth two-zone field on the cell grid (width = w_cells cells)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    arg = (X - (p[2] + p[3] * (Y - 0.5))) / (w_cells * DX / LX)
    sig = 1.0 / (1.0 + xp.exp(-arg))
    lo, hi = xp.exp(p[0]), xp.exp(p[1])
    return lo + (hi - lo) * sig

def cf_true_field():
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    return np.where(X < B0_TRUE + S_TRUE * (Y - 0.5), CF_LOW_TRUE, CF_HIGH_TRUE)


# ---- K-zone model: K strips along x separated by K-1 parallel (common-slope) smooth boundaries ----
def cf_zonal_K(xp, theta, K, w_cells=0.5):
    """theta = [ln c_1..ln c_K, b_1..b_{K-1}, s]  (K=1: [ln c_1]).  Boundary k: x/LX = b_k + s*(y/LY-0.5)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    f = xp.exp(theta[0]) * xp.ones((NX, NY))
    for k in range(K - 1):
        arg = (X - (theta[K + k] + theta[2 * K - 1] * (Y - 0.5))) / (w_cells * DX / LX)
        f = f + (xp.exp(theta[k + 1]) - xp.exp(theta[k])) / (1.0 + xp.exp(-arg))
    return f

# ---- JAX forward model returning only the observed cells (keeps memory tiny) ----
def make_forward(U0, ox, oy):
    oxj, oyj = jnp.array(ox), jnp.array(oy)
    def fwd(Cf):
        def body(state, n):
            new = swe_step(jnp, state, n * DT, Cf, U0)
            e, u, v = new
            uc = 0.5 * (u[:-1, :] + u[1:, :]); vc = 0.5 * (v[:, :-1] + v[:, 1:])
            return new, (e[oxj, oyj], uc[oxj, oyj], vc[oxj, oyj])
        init = (jnp.zeros((NX, NY)), jnp.zeros((NX + 1, NY)), jnp.zeros((NX, NY + 1)))
        _, ys = lax.scan(body, init, jnp.arange(CFG["N_STEPS"]))
        return ys
    return fwd

CF_T = cf_true_field()
BOUNDS = [(np.log(5e-4), np.log(2e-2))] * 2 + [(0.1, 0.9), (-0.5, 0.5)]
P_TRUE = np.array([np.log(CF_LOW_TRUE), np.log(CF_HIGH_TRUE), B0_TRUE, S_TRUE])

def build_problem(U0, regime, variant, P_use, noise_seed=0, cf_truth=None):
    """Truth from the independent NumPy loop; inversion uses the JAX solver (same equations => 'inverse crime',
    acceptable for an identifiability test, NOT for validation). Layout and noise change with noise_seed."""
    cf_truth = CF_T if cf_truth is None else cf_truth
    rng = np.random.default_rng(CFG["SEED"] + noise_seed)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    ox, oy = np.unravel_index(cells[:P_use], (NX, NY))
    t_idx = REGIMES[regime]
    Hc, Uc, Vc = run_obs_np(cf_truth, U0, CFG["N_STEPS"], ox, oy)
    clean = dict(h=Hc[t_idx], u=Uc[t_idx], v=Vc[t_idx])
    sd = {k: CFG["NOISE_FRAC"] * clean[k].std() for k in clean}
    nr = np.random.default_rng(11 + noise_seed)
    obs = {k: clean[k] + nr.normal(scale=sd[k], size=clean[k].shape) for k in clean}
    fwd = make_forward(U0, ox, oy)
    tj = jnp.array(t_idx)
    def chi2(Cf):
        H, U, V = fwd(Cf)
        c = jnp.mean(((H[tj] - obs["h"]) / sd["h"]) ** 2)
        if variant == "huv":
            c = (c + jnp.mean(((U[tj] - obs["u"]) / sd["u"]) ** 2) + jnp.mean(((V[tj] - obs["v"]) / sd["v"]) ** 2)) / 3.0
        return c
    return dict(U0=U0, regime=regime, variant=variant, P=P_use, ox=ox, oy=oy, t_idx=t_idx, obs=obs, sd=sd,
                fwd=fwd, chi2=chi2, n_obs=int(obs["h"].size * (3 if variant == "huv" else 1)))

## Cell 3 — GATE (must pass before interpreting anything)

In [3]:
gate = {}
pb = build_problem(CFG["U0_LIST"][-1], "rich", "h", CFG["P_OBS"])
H_j, U_j, V_j = [np.asarray(a) for a in pb["fwd"](jnp.array(CF_T))]
H_n, U_n, V_n = run_obs_np(CF_T, pb["U0"], CFG["N_STEPS"], pb["ox"], pb["oy"])
rel = np.max(np.abs(H_j - H_n)) / np.std(H_n)
print(f"(a) JAX vs NumPy solver: max|dh|/std(h) = {rel:.2e}  ->", "PASS" if rel < 1e-8 else "FAIL"); gate["solver_match"] = bool(rel < 1e-8)
loss_p = lambda p: pb["chi2"](cf_zonal(jnp, p))
vg = jax.jit(jax.value_and_grad(loss_p))
p0 = P_TRUE + np.array([0.2, -0.2, 0.03, -0.02])
v0, g0 = vg(jnp.array(p0)); g0 = np.asarray(g0)
fd = np.zeros(4)
for i in range(4):
    e = np.zeros(4); e[i] = 1e-6
    fd[i] = (float(loss_p(jnp.array(p0 + e))) - float(loss_p(jnp.array(p0 - e)))) / 2e-6
err = np.max(np.abs(g0 - fd) / (np.abs(fd) + 1e-12))
print(f"(b) grad check: max rel err = {err:.2e} ->", "PASS" if err < 1e-3 else "FAIL"); gate["grad_ok"] = bool(err < 1e-3)
c_ref = float(loss_p(jnp.array(P_TRUE)))
print(f"(c) chi2 at true params (h-only, U0={pb['U0']}): {c_ref:.3f} (expect ~1.0 +/- {2*np.sqrt(2/pb['n_obs']):.3f})")
gate["chi2_ok"] = bool(abs(c_ref - 1) < 4 * np.sqrt(2 / pb["n_obs"]))
gcf = np.asarray(jax.grad(pb["chi2"])(jnp.array(CF_T)))
print(f"(d) informational: d chi2 / d Cf[0,0] = {gcf[0,0]:.1e} (structural null cell: never enters the solver stencil); max|d chi2/d Cf| = {np.abs(gcf).max():.1e}")
assert all(gate.values()), "GATE FAILED — do not interpret anything below; send this cell's output to the assistant."

(a) JAX vs NumPy solver: max|dh|/std(h) = 8.07e-15  -> PASS
(b) grad check: max rel err = 7.98e-09 -> PASS
(c) chi2 at true params (h-only, U0=1.0): 0.999 (expect ~1.0 +/- 0.062)
(d) informational: d chi2 / d Cf[0,0] = 0.0e+00 (structural null cell: never enters the solver stencil); max|d chi2/d Cf| = 6.1e-01


## Cell 4 — Zonal 4-parameter model: multi-start, several noise realisations
Verdict per realisation: **identified** (≥1 start at noise level AND every noise-level start has boundary error ≤ 1 cell and both Cf within 25 %), **not identified** (noise-level starts exist but disagree / are far from truth), **optimiser failed** (no start reaches noise level — says nothing about information).

In [4]:
def fit_zonal(pb, p0):
    vgf = jax.jit(jax.value_and_grad(lambda p: pb["chi2"](cf_zonal(jnp, p))))
    def f(p):
        v, g = vgf(jnp.array(p)); return float(v), np.asarray(g, dtype=np.float64)
    res = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=BOUNDS,
                   options=dict(maxiter=CFG["ZONAL_MAXITER"], ftol=1e-12, gtol=1e-8))
    p = res.x
    bnd_km = float(np.mean(np.abs((p[2] - B0_TRUE) + (p[3] - S_TRUE) * (YC - 0.5))) * LX / 1e3)
    return dict(cf_low=float(np.exp(p[0])), cf_high=float(np.exp(p[1])), b0=float(p[2]), s=float(p[3]),
                chi2=float(res.fun), nit=int(res.nit), bnd_km=bnd_km)

def verdict(sols, chi2_ref, n_obs):
    tol = 2 * np.sqrt(2 / n_obs)
    equiv = [s for s in sols if s["chi2"] <= chi2_ref + tol]
    if not equiv: return "optimiser_failed", equiv
    ok = all(s["bnd_km"] <= CFG["TOL_BND_KM"] and abs(s["cf_low"] / CF_LOW_TRUE - 1) <= CFG["TOL_CF_REL"]
             and abs(s["cf_high"] / CF_HIGH_TRUE - 1) <= CFG["TOL_CF_REL"] for s in equiv)
    return ("identified" if ok else "not_identified"), equiv

RESULTS = {"zonal": [], "free": [], "jac": [], "ksweep": [], "mode": MODE,
           "cfg": {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in CFG.items()}}
ZON_CONFIGS = [("h", CFG["P_OBS"]), ("huv", CFG["P_OBS"] // 3)]       # equal observation count (39 vs 3x13)
t0 = time.time()
for U0 in CFG["U0_LIST"]:
    for regime in REGIMES:
        for variant, P_use in ZON_CONFIGS:
            per_seed = []
            for ns in range(CFG["N_NOISE"]):
                pbz = build_problem(U0, regime, variant, P_use, noise_seed=ns)
                chi2_ref = float(pbz["chi2"](cf_zonal(jnp, jnp.array(P_TRUE))))
                sols = []
                for k in range(CFG["N_STARTS"]):
                    r = np.random.default_rng(100 + k + 17 * ns)
                    p0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4),
                                   r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
                    sols.append(fit_zonal(pbz, p0))
                v, equiv = verdict(sols, chi2_ref, pbz["n_obs"])
                per_seed.append(dict(noise_seed=ns, verdict=v, chi2_ref=chi2_ref, best_chi2=min(s["chi2"] for s in sols),
                                     n_equiv=len(equiv), n_starts=len(sols),
                                     bnd_km_worst_equiv=(max(s["bnd_km"] for s in equiv) if equiv else None), sols=sols))
            cnt = lambda name: sum(1 for d in per_seed if d["verdict"] == name)
            worst = [d["bnd_km_worst_equiv"] for d in per_seed if d["bnd_km_worst_equiv"] is not None]
            rec = dict(U0=U0, regime=regime, variant=variant, n_obs=pbz["n_obs"], n_noise=len(per_seed),
                       identified=cnt("identified"), not_identified=cnt("not_identified"), optimiser_failed=cnt("optimiser_failed"),
                       median_worst_bnd_km=(float(np.median(worst)) if worst else None), per_seed=per_seed)
            RESULTS["zonal"].append(rec)
            print(f"[zonal] U0={U0:<4} {regime:8s} {variant:3s} nobs={pbz['n_obs']:5d} | identified {rec['identified']}/{rec['n_noise']}  "
                  f"not-identified {rec['not_identified']}  optimiser-failed {rec['optimiser_failed']} | median worst boundary err among noise-level starts: "
                  f"{('%.1f km' % rec['median_worst_bnd_km']) if worst else 'n/a'}  [{time.time()-t0:.0f}s]")
print("truth: Cf_low=%.5f Cf_high=%.5f b0=%.2f s=%.2f | tolerances: boundary <= %.2f km, Cf within %d%%" %
      (CF_LOW_TRUE, CF_HIGH_TRUE, B0_TRUE, S_TRUE, CFG["TOL_BND_KM"], 100 * CFG["TOL_CF_REL"]))

[zonal] U0=0.15 rich     h   nobs= 2106 | identified 0/3  not-identified 3  optimiser-failed 0 | median worst boundary err among noise-level starts: 32.0 km  [79s]
[zonal] U0=0.15 rich     huv nobs= 2106 | identified 0/3  not-identified 3  optimiser-failed 0 | median worst boundary err among noise-level starts: 32.5 km  [164s]
[zonal] U0=0.15 sparse4  h   nobs=  156 | identified 0/3  not-identified 3  optimiser-failed 0 | median worst boundary err among noise-level starts: 28.3 km  [256s]
[zonal] U0=0.15 sparse4  huv nobs=  156 | identified 0/3  not-identified 3  optimiser-failed 0 | median worst boundary err among noise-level starts: 22.5 km  [343s]
[zonal] U0=1.0  rich     h   nobs= 2106 | identified 0/3  not-identified 3  optimiser-failed 0 | median worst boundary err among noise-level starts: 47.8 km  [427s]
[zonal] U0=1.0  rich     huv nobs= 2106 | identified 3/3  not-identified 0  optimiser-failed 0 | median worst boundary err among noise-level starts: 0.5 km  [510s]
[zonal] U0=1

## Cell 5 — Free 288-cell field: λ continuation (large → small, warm-started)
λ is chosen by the **discrepancy principle** (largest λ whose data-χ² is at noise level — uses no truth); the *oracle* best λ (max r) is also marked so that the effect of regularisation choice is visible. Rows with `conv=False` hit the iteration cap — treat r/RMSE there as lower-quality estimates.

In [5]:
def fit_free_path(pb, lams):
    q = np.full(NX * NY, np.log(2.5e-3)); rows = []
    for lam in sorted(lams, reverse=True):
        def obj(qv, lam=lam):
            lnc = qv.reshape(NX, NY)
            smooth = jnp.mean((lnc[1:, :] - lnc[:-1, :]) ** 2) + jnp.mean((lnc[:, 1:] - lnc[:, :-1]) ** 2)
            return pb["chi2"](jnp.exp(lnc)) + lam * smooth
        vgf = jax.jit(jax.value_and_grad(obj))
        def f(qv):
            v, g = vgf(jnp.array(qv)); return float(v), np.asarray(g, dtype=np.float64)
        res = minimize(f, q, jac=True, method="L-BFGS-B", bounds=[(np.log(5e-4), np.log(2e-2))] * (NX * NY),
                       options=dict(maxiter=CFG["FREE_MAXITER"], ftol=1e-12, gtol=1e-8))
        q = res.x; cf = np.exp(q).reshape(NX, NY)
        rows.append(dict(lam=lam, r=float(np.corrcoef(cf.ravel(), CF_T.ravel())[0, 1]),
                         rmse_pct=float(100 * np.sqrt(np.mean((cf - CF_T) ** 2)) / (CF_T.max() - CF_T.min())),
                         chi2_data=float(pb["chi2"](jnp.array(cf))), cf_std=float(cf.std()), nit=int(res.nit),
                         conv=bool(res.nit < CFG["FREE_MAXITER"])))
    tol = 1 + 2 * np.sqrt(2 / pb["n_obs"])
    ok = [i for i, r_ in enumerate(rows) if r_["chi2_data"] <= tol]
    dp = ok[0] if ok else len(rows) - 1                      # rows are in descending lambda
    oracle = int(np.argmax([r_["r"] for r_ in rows]))
    return rows, dp, oracle

t0 = time.time()
for U0 in CFG["U0_LIST"]:
    for regime in REGIMES:
        for variant, P_use in ZON_CONFIGS:
            for ns in range(CFG["N_NOISE_FREE"]):
                pbf = build_problem(U0, regime, variant, P_use, noise_seed=ns)
                rows, dp, orc = fit_free_path(pbf, CFG["LAMBDAS"])
                RESULTS["free"].append(dict(U0=U0, regime=regime, variant=variant, noise_seed=ns, n_obs=pbf["n_obs"], rows=rows, dp_index=dp, oracle_index=orc))
                print(f"[free ] U0={U0:<4} {regime:8s} {variant:3s} ns={ns} nobs={pbf['n_obs']:5d} [{time.time()-t0:.0f}s]")
                for i, r_ in enumerate(rows):
                    tag = ("DP " if i == dp else "   ") + ("ORC" if i == orc else "   ")
                    print(f"        {tag} lam={r_['lam']:.0e}  r={r_['r']:6.3f}  RMSE={r_['rmse_pct']:6.1f}%  chi2_data={r_['chi2_data']:.3f}  std(Cf_hat)={r_['cf_std']:.5f}  nit={r_['nit']}  conv={r_['conv']}")

[free ] U0=0.15 rich     h   ns=0 nobs= 2106 [84s]
        DP     lam=1e-01  r=-0.422  RMSE=  69.1%  chi2_data=0.991  std(Cf_hat)=0.00077  nit=52  conv=True
               lam=1e-02  r=-0.185  RMSE= 121.2%  chi2_data=0.985  std(Cf_hat)=0.00254  nit=88  conv=True
           ORC lam=1e-03  r=-0.026  RMSE= 208.9%  chi2_data=0.980  std(Cf_hat)=0.00503  nit=200  conv=True
               lam=1e-04  r=-0.046  RMSE= 235.3%  chi2_data=0.979  std(Cf_hat)=0.00567  nit=250  conv=False
[free ] U0=0.15 rich     huv ns=0 nobs= 2106 [222s]
        DP ORC lam=1e-01  r= 0.754  RMSE=  33.1%  chi2_data=0.972  std(Cf_hat)=0.00099  nit=250  conv=False
               lam=1e-02  r= 0.305  RMSE=  94.2%  chi2_data=0.949  std(Cf_hat)=0.00241  nit=250  conv=False
               lam=1e-03  r= 0.189  RMSE= 142.1%  chi2_data=0.941  std(Cf_hat)=0.00357  nit=250  conv=False
               lam=1e-04  r= 0.176  RMSE= 155.1%  chi2_data=0.940  std(Cf_hat)=0.00390  nit=250  conv=False
[free ] U0=0.15 sparse4  h   ns=0 nobs

## Cell 6 — Information content of the free field: noise-normalised Jacobian modes (equal observation count)
Reported: number of singular values σ of `J/σ_noise` above 1, 3 and 10 (σ>1 ≈ a unit-norm Cf perturbation of 0.06×CF_SCALE per cell is detectable at 1σ). The earlier `rank(1e-6)`/`cond` columns are **removed**: Cf[0,0] is structurally unobservable (exact null direction) and relative-threshold ranks are insensitive to noise.

In [6]:
def jac_matrix(pb, fmap, x0, chunk=36):
    t_j = jnp.array(pb["t_idx"]); sd = pb["sd"]; fwd = pb["fwd"]; variant = pb["variant"]
    def f(x):
        H, U, V = fwd(fmap(x))
        out = [(H[t_j] / sd["h"]).ravel()]
        if variant == "huv":
            out += [(U[t_j] / sd["u"]).ravel(), (V[t_j] / sd["v"]).ravel()]
        return jnp.concatenate(out)
    n = int(x0.shape[0]); x0 = jnp.array(x0)
    jvp_one = jax.jit(jax.vmap(lambda tv: jax.jvp(f, (x0,), (tv,))[1]))
    eye = jnp.eye(n)
    cols = [np.asarray(jvp_one(eye[i:i + chunk])) for i in range(0, n, chunk)]
    return np.concatenate(cols, axis=0).T                     # (n_obs, n)

t0 = time.time()
for U0 in CFG["U0_LIST"]:
    for variant, P_use in [("h", CFG["P_OBS"]), ("huv", CFG["P_OBS"] // 3), ("huv", CFG["P_OBS"])]:
        pbj = build_problem(U0, "rich", variant, P_use)
        J = jac_matrix(pbj, lambda x: (x * CFG["CF_SCALE"]).reshape(NX, NY), CF_T.ravel() / CFG["CF_SCALE"])
        s = np.linalg.svd(J, compute_uv=False)
        rec = dict(U0=U0, variant=variant, P=P_use, n_obs=pbj["n_obs"], modes_gt1=int((s > 1).sum()), modes_gt3=int((s > 3).sum()),
                   modes_gt10=int((s > 10).sum()), sv_top=float(s[0]))
        RESULTS["jac"].append(rec)
        print(f"[jac  ] U0={U0:<4} {variant:3s} P={P_use:2d} nobs={pbj['n_obs']:5d} | modes sigma>1: {rec['modes_gt1']:3d}/288  >3: {rec['modes_gt3']:3d}  >10: {rec['modes_gt10']:3d}  [{time.time()-t0:.0f}s]")

[jac  ] U0=0.15 h   P=39 nobs= 2106 | modes sigma>1:   2/288  >3:   1  >10:   0  [6s]
[jac  ] U0=0.15 huv P=13 nobs= 2106 | modes sigma>1:  48/288  >3:  24  >10:  12  [12s]
[jac  ] U0=0.15 huv P=39 nobs= 6318 | modes sigma>1: 110/288  >3:  65  >10:  36  [19s]
[jac  ] U0=1.0  h   P=39 nobs= 2106 | modes sigma>1:  10/288  >3:   2  >10:   1  [25s]
[jac  ] U0=1.0  huv P=13 nobs= 2106 | modes sigma>1: 133/288  >3:  77  >10:  32  [31s]
[jac  ] U0=1.0  huv P=39 nobs= 6318 | modes sigma>1: 206/288  >3: 151  >10:  83  [38s]


## Cell 7 — Identifiability–complexity curve (H1d) — Cramér–Rao prediction for a K-zone model
For each K the *reference* medium has K strips with alternating Cf (1.5e-3 / 4e-3), equally spaced boundaries and slope 0.1. Parameters θ = (ln c₁…ln c_K, b₁…b_{K−1}, s), scaled so that **1 unit = 100 % in Cf or 1 grid cell in boundary position**. From the noise-normalised Jacobian we get the Cramér–Rao standard deviations (a *lower bound*: real estimators do no better). A K is **predicted identifiable** if the worst boundary σ ≤ 1 cell and the worst Cf σ ≤ 25 %. No fitting is involved; this predicts where the zonal curve should break and can be compared with real fits later.

In [7]:
def ref_theta(K):
    c = np.array([1.5e-3 if k % 2 == 0 else 4.0e-3 for k in range(K)]) if K > 1 else np.array([2.5e-3])
    if K == 1: return np.log(c), 1
    return np.concatenate([np.log(c), np.arange(1, K) / K, [0.1]]), 2 * K

t0 = time.time()
for U0 in CFG["U0_LIST"]:
    for regime in REGIMES:
        for variant, P_use in ZON_CONFIGS:
            for K in CFG["K_LIST"]:
                th, n = ref_theta(K)
                cf_ref = np.asarray(cf_zonal_K(np, th, K))
                scale = np.concatenate([np.ones(K), np.full(n - K, 1.0 / NX)]) if K > 1 else np.ones(1)
                pbk = build_problem(U0, regime, variant, P_use, cf_truth=cf_ref)
                J = jac_matrix(pbk, lambda d: cf_zonal_K(jnp, th + d * scale, K), np.zeros(n), chunk=n)
                F = J.T @ J; w = np.linalg.eigvalsh(F)
                sv = np.sqrt(np.clip(w, 0, None))
                singular = bool(w.min() <= 1e-10 * w.max())
                if singular:
                    sd_cf, sd_bnd = np.inf, np.inf
                else:
                    sd_all = np.sqrt(np.diag(np.linalg.inv(F)))
                    sd_cf = float(sd_all[:K].max()); sd_bnd = float(sd_all[K:].max() * DX / 1e3) if K > 1 else 0.0
                pred = (not singular) and sd_cf <= CFG["TOL_CF_REL"] and sd_bnd <= CFG["TOL_BND_KM"]
                rec = dict(U0=U0, regime=regime, variant=variant, K=K, n_params=n, modes_gt1=int((sv > 1).sum()), singular=singular,
                           crb_cf_rel_max=sd_cf, crb_bnd_km_max=sd_bnd, predicted_identifiable=bool(pred))
                RESULTS["ksweep"].append(rec)
                print(f"[K    ] U0={U0:<4} {regime:8s} {variant:3s} K={K} params={n:2d} | modes>1: {rec['modes_gt1']:2d} | CRB worst Cf {sd_cf*100:8.1f}%  worst boundary {sd_bnd:8.2f} km | "
                      f"{'PREDICTED IDENTIFIABLE' if pred else 'not identifiable'}  [{time.time()-t0:.0f}s]")

[K    ] U0=0.15 rich     h   K=1 params= 1 | modes>1:  1 | CRB worst Cf      1.8%  worst boundary     0.00 km | PREDICTED IDENTIFIABLE  [1s]
[K    ] U0=0.15 rich     h   K=2 params= 4 | modes>1:  3 | CRB worst Cf     37.0%  worst boundary    54.59 km | not identifiable  [3s]
[K    ] U0=0.15 rich     h   K=3 params= 6 | modes>1:  3 | CRB worst Cf     89.5%  worst boundary    24.90 km | not identifiable  [5s]
[K    ] U0=0.15 rich     h   K=4 params= 8 | modes>1:  4 | CRB worst Cf    145.4%  worst boundary    47.11 km | not identifiable  [6s]
[K    ] U0=0.15 rich     h   K=6 params=12 | modes>1:  5 | CRB worst Cf    313.5%  worst boundary    42.90 km | not identifiable  [8s]
[K    ] U0=0.15 rich     huv K=1 params= 1 | modes>1:  1 | CRB worst Cf      0.8%  worst boundary     0.00 km | PREDICTED IDENTIFIABLE  [9s]
[K    ] U0=0.15 rich     huv K=2 params= 4 | modes>1:  4 | CRB worst Cf     11.8%  worst boundary     3.11 km | PREDICTED IDENTIFIABLE  [11s]
[K    ] U0=0.15 rich     huv K=3 par

## Cell 8 — Save and one-screen summary (paste this output back)

In [8]:
def _default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if hasattr(o, "tolist"): return o.tolist()
    return str(o)
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
path = os.path.join(out_dir, "control_results_v2.json")
json.dump(RESULTS, open(path, "w"), indent=1, default=_default)
print("saved ->", path, "| MODE =", MODE)
print("\n=== ZONAL 4-param (verdicts over noise realisations) ===")
for r in RESULTS["zonal"]:
    print(f"U0={r['U0']:<4} {r['regime']:8s} {r['variant']:3s} | identified {r['identified']}/{r['n_noise']}  not-identified {r['not_identified']}  optimiser-failed {r['optimiser_failed']}"
          f" | median worst bnd err {('%.1f km' % r['median_worst_bnd_km']) if r['median_worst_bnd_km'] is not None else 'n/a'}")
print("\n=== FREE FIELD (discrepancy-principle lambda | oracle lambda) ===")
for r in RESULTS["free"]:
    d, o = r["rows"][r["dp_index"]], r["rows"][r["oracle_index"]]
    print(f"U0={r['U0']:<4} {r['regime']:8s} {r['variant']:3s} ns={r['noise_seed']} | DP: r={d['r']:.3f} RMSE={d['rmse_pct']:.0f}% conv={d['conv']} | oracle: r={o['r']:.3f} RMSE={o['rmse_pct']:.0f}%")
print("\n=== JACOBIAN information modes (sigma>1) ===")
for r in RESULTS["jac"]:
    print(f"U0={r['U0']:<4} {r['variant']:3s} nobs={r['n_obs']:5d} | {r['modes_gt1']:3d}/288")
print("\n=== K-ZONE CRB prediction (rows with variant h, regime rich) ===")
for r in RESULTS["ksweep"]:
    if r["variant"] == "h" and r["regime"] == "rich":
        print(f"U0={r['U0']:<4} K={r['K']} params={r['n_params']:2d} modes>1={r['modes_gt1']:2d} | worst Cf {r['crb_cf_rel_max']*100:8.1f}%  worst bnd {r['crb_bnd_km_max']:8.2f} km | {'identifiable' if r['predicted_identifiable'] else 'NOT'}")

saved -> /kaggle/working/control_results_v2.json | MODE = standard

=== ZONAL 4-param (verdicts over noise realisations) ===
U0=0.15 rich     h   | identified 0/3  not-identified 3  optimiser-failed 0 | median worst bnd err 32.0 km
U0=0.15 rich     huv | identified 0/3  not-identified 3  optimiser-failed 0 | median worst bnd err 32.5 km
U0=0.15 sparse4  h   | identified 0/3  not-identified 3  optimiser-failed 0 | median worst bnd err 28.3 km
U0=0.15 sparse4  huv | identified 0/3  not-identified 3  optimiser-failed 0 | median worst bnd err 22.5 km
U0=1.0  rich     h   | identified 0/3  not-identified 3  optimiser-failed 0 | median worst bnd err 47.8 km
U0=1.0  rich     huv | identified 3/3  not-identified 0  optimiser-failed 0 | median worst bnd err 0.5 km
U0=1.0  sparse4  h   | identified 3/3  not-identified 0  optimiser-failed 0 | median worst bnd err 3.1 km
U0=1.0  sparse4  huv | identified 3/3  not-identified 0  optimiser-failed 0 | median worst bnd err 1.5 km

=== FREE FIELD (discr